# 16장 최종 시연

이 노트북은 웹 교재 [16장 최종 시연](https://pidocs.aifusionlab.org/ai-python/ch16/)의 본문 코드를 절 순서대로 담고 있습니다. 위에서부터 차례로 실행하면서 본문의 실행 결과와 비교합니다.

## 16.2 시연 점검 기준

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch16/16-2/)

In [ ]:
import pandas as pd

demo = pd.DataFrame({
    "회차": [1, 1, 1, 2, 2, 2, 3, 3, 3],
    "조작": ["조용한 상태", "대화", "손뼉"] * 3,
    "기대": ["정상", "주의", "경고"] * 3,
    "표시": ["정상", "주의", "경고", "정상", "정상", "경고", "정상", "주의", "경고"],
})
demo["일치"] = demo["기대"] == demo["표시"]
print(demo)
print("일치율:", round(demo["일치"].mean(), 3))

In [ ]:
order = ["정상", "주의", "경고"]
ct = pd.crosstab(pd.Categorical(demo["기대"], order), pd.Categorical(demo["표시"], order),
                 rownames=["기대"], colnames=["표시"])
ct

In [ ]:
import re

PATTERNS = {
    "Google API 키": r"AIza[0-9A-Za-z_\-]{30,}",
    "Wi-Fi 비밀번호": r'WIFI_PASS\s*=\s*"(?!교수자가)[^"]+"',
}

def check_secrets(code):
    found = []
    for no, line in enumerate(code.splitlines(), 1):
        for name, pat in PATTERNS.items():
            if re.search(pat, line):
                found.append((no, name))
    return found

safe = '''WIFI_SSID = "교수자가 안내한 Wi-Fi 이름"
WIFI_PASS = "교수자가 안내한 비밀번호"
key = userdata.get("GEMINI_API_KEY")'''
unsafe = 'WIFI_PASS = "abcd1234"\nclient = genai.Client(api_key="' + "AIza" + "x" * 35 + '")'

print("safe:", check_secrets(safe))
print("unsafe:", check_secrets(unsafe))

In [ ]:
import glob

for path in glob.glob("*.py"):
    result = check_secrets(open(path, encoding="utf-8").read())
    print(path, result if result else "이상 없음")

## 16.3 회고와 개선 과제

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch16/16-3/)

In [ ]:
import pandas as pd

log = pd.DataFrame({
    "회차": [1] * 4 + [2] * 4 + [3] * 4,
    "항목": ["센서·기기", "Wi-Fi·전송", "서버", "AI 보고"] * 3,
    "결과": ["PASS", "FAIL", "PASS", "FAIL",
             "PASS", "FAIL", "PASS", "PASS",
             "PASS", "PASS", "PASS", "PASS"],
})
print(log.pivot(index="항목", columns="회차", values="결과"))
fails = log[log["결과"] == "FAIL"].groupby("항목").size().sort_values(ascending=False)
print("\nFAIL 횟수")
print(fails)